# Preparation cost and state windows

This notebook follows a complete research task: bound a preparation's cost, check candidate programs, examine subsystem structure, and pass the saved calculation to an independent reader. The examples use explicit reset and cooling models, followed by a Bell-pair/GHZ comparison.

Start with the [0.3 source quickstart](README.md#quick-start) and [notebook setup](README.md#jupyter-notebook). Select **Python (rcc-refcert)** as the kernel, then run the cells in order. The published 0.2.0 package covers the model and terminal-state tutorials; this notebook uses the cost and window workflows.

You can change the requests below and rerun them. The [workflow guide](docs/COST_AND_WINDOWS.md) describes exact parameters, your own matrices, tensor labels and numerical evidence.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from rcc_refcert.bundle import verify_analysis_bundle
from rcc_refcert.workflows import (
    analyze_cost,
    analyze_windows,
    export_probe,
    load_workflow_template,
    render_workflow_result,
    save_workflow_bundle,
)

## Bound a complete preparation task

The cyclic-reset template fixes a four-qubit model, a complete target in its original basis, and candidate programs. Each physical action costs one atomic slot, including halt. An exact reference-envelope calculation excludes smaller budgets; executing a candidate establishes a feasible upper endpoint when it reaches the target tolerance.

The scan below changes only the allowed trace distance:

| Tolerance | Cost interval | Interpretation |
|---|---|---|
| `1/10` | `[5,5]` | Five slots are necessary and sufficient in this model. |
| `9/20` | `[4,4]` | The four-slot candidate now meets the target. |
| `11/25` | `[4,5]` | The lower bound and best supplied candidate leave a one-slot gap. |

A candidate can miss the target while the analysis completes successfully. To explore a different physical instance, edit the template's `qubits` and `p` parameters before calling `analyze_cost`; use exact strings such as `"19/20"` for probabilities.

In [ ]:
request = load_workflow_template("cyclic-reset")
request["epsilon_values"] = ["1/10", "9/20", "11/25"]
result = analyze_cost(request)
print(render_workflow_result(result))
assert [p["cost"]["interval_slots"] for p in result["epsilon_scan"]] == [
    [5, 5],
    [4, 4],
    [4, 5],
]

## Locate structure with subsystem windows

The first probe compares paired Bell states with a four-qubit GHZ state. At the default parameters, both are globally pure. With `Gamma=4`, their global RCC main term is 2; windows 12, 13 and 34 give `(1,0,1)` for the Bell pairs and `(1/2,1/2,1/2)` for GHZ. The cross window shows whether the same global structure is concentrated inside a pair or shared across the register.

Each window reports the reduced-state entropy `S(rho_A)`, its entropy deficit and the deficit divided by `log2(Gamma)`. The corresponding full-space effective state is `rho_A ⊗ I_B/d_B`; its entropy includes the additional `log2(d_B)` from the replaced complement. These are state-side quantities. A preparation-cost bound also uses model and accuracy information.

The cell compares the pure-state probe with a common uniform-noise mixture. Changing `mixing` keeps the two global spectra equal but changes the window profiles; `gamma` sets the declared comparison scale. The collection also includes `w2-noise-distribution`, which distributes phase noise differently at matched global entropy. Window entropies are numerical, with input and precision diagnostics retained.


In [ ]:
windows = analyze_windows(export_probe("w1-quantum-organization"))
print(render_workflow_result(windows))

mixed_windows = analyze_windows(
    export_probe("w1-quantum-organization", parameters={"mixing": "1/4"})
)
for profile in mixed_windows["profiles"]:
    values = [round(q["rcc_main_term"], 6) for q in profile["windows"]]
    print(profile["id"], "window C after uniform mixing:", values)

## Follow structure along a cooling preparation

The cooling probe refreshes two bath qubits and applies reversible compression to concentrate purity on the target qubit. Its windows follow the actual candidate execution, with the action and cumulative cost shown at each step.

At tolerance `1/50`, the gain bound excludes costs below six and the six-action candidate reaches the complete target within tolerance, giving a rigorous `[6,6]` cost interval. The eight-action target program supplies the target itself. The reference scale is `g_R=log2(3)`, derived from this model's action classes.

Inspect refresh steps to see entropy and target–bath correlation removed; compression redistributes the deficit at a fixed global spectrum. The states are diagonal, so the reported correlation is classical. Halt adds one charged slot and leaves the state unchanged. To compare candidate improvements, hold the model, full target and tolerance fixed.


In [ ]:
cooling_request = export_probe("w3-cooling-trajectory")
cooling = analyze_cost(cooling_request)
print(render_workflow_result(cooling))
assert cooling["cost"]["interval_slots"] == [6, 6]
print(render_workflow_result(cooling["windows"]))

## Share a calculation for independent verification

A workflow bundle saves the full effective request and the author's result. Verification reconstructs the model, checks the evidence and reruns the calculation, assigning the recipient's result a new run identifier. The original evidence remains available for comparison.

Each run of the next cell creates a new persistent folder under `generated_results/workflow-notebook/`, prints the full bundle path and verifies it independently. The file remains there after the notebook closes, and earlier bundles are preserved. Share this file with the producer version; the recipient can run `python -m rcc_refcert bundle verify PATH`.

`historical_results_supported` should be `True`. Cost and window bundles require the 0.3 reader; the [bundle guide](docs/ANALYSIS_BUNDLES.md#complete-cost-and-window-workflows) explains the result fields. To apply the workflows to your own matrices or channels, follow the [complete numerical model example](examples/numerical_cost_workflow.py).


In [ ]:
output_root = Path("generated_results") / "workflow-notebook"
output_root.mkdir(parents=True, exist_ok=True)
output_directory = Path(mkdtemp(prefix="cost-", dir=output_root))
bundle = output_directory / "cost.rccbundle"
save_workflow_bundle(bundle, request, result)
verification = verify_analysis_bundle(bundle)
assert verification["historical_results_supported"] is True
assert verification["fresh_result"]["run_id"] != result["run_id"]
print("Saved bundle:", bundle.resolve())
print("Fresh verification:", verification["fresh_verification"]["outcome"])
print("Historical results supported:", verification["historical_results_supported"])